<a href="https://colab.research.google.com/github/wuhao007/haowu999/blob/main/btc_haowu999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ₿ Bitcoin AHR999 深度研究工具
**Haowu999 Quant Studio**

本 Notebook 提供 BTC 的完整 AHR999 分析套件，包括：
- 🔬 幂律回归拟合 + 模型置信度评级
- 📊 减半周期叠加分析
- 💰 历史回测模拟（买入/卖出信号验证）
- 📈 多周期幂律衰减对比
- 🖥️ Plotly 交互式仪表盘
- ⚡ 增量数据缓存（首次全量下载，后续秒级增量更新）

In [ ]:
!pip install yfinance plotly seaborn scikit-learn matplotlib --quiet
import datetime
import os
import time
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import yfinance as yf
import unittest
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
print('✅ All dependencies loaded.')


In [ ]:
_AHR999_DAYS = 200

# === Bitcoin 减半事件时间表 ===
HALVING_EVENTS = [
    ("2012-11-28", "1st Halving (50→25)"),
    ("2016-07-09", "2nd Halving (25→12.5)"),
    ("2020-05-11", "3rd Halving (12.5→6.25)"),
    ("2024-04-19", "4th Halving (6.25→3.125)"),
]

# === 默认缓存路径 ===
# Colab: /content/btc_cache.csv (临时, 但单次会话内有效)
# 本地:  当前目录下的 btc_cache.csv
# Google Drive: /content/drive/MyDrive/btc_cache.csv (持久化)
_DEFAULT_CACHE_DIR = os.getcwd()

class Haowu999:
    """
    AHR999 定投/抄底指数 — 增强版 (含增量缓存)
    公式: ahr999 = (币价 / 200日均价) × (币价 / 拟合公允价)
    
    增强功能:
    - ⚡ 增量数据缓存: 首次全量下载 → 保存 CSV → 后续只下载最新数据
    - 信噪比 (SNR) 计算
    - 模型置信度评级 (A+/A/B/C)
    - 卖出信号线 (AHR999x = 5.0)
    - 多周期拟合对比
    """

    def __init__(self, start_date='2009-01-03', coin='BTC-USD', cache_dir=None):
        self.coin = coin
        self.start_date = pd.to_datetime(start_date)
        self.prices = None
        self.w = None
        self.b = None
        self.r2 = None
        self.dates = None
        self.ydata = None
        self.predicted_ydata = None
        
        # 缓存配置
        cache_base = cache_dir or _DEFAULT_CACHE_DIR
        safe_coin = coin.replace('=', '_').replace('-', '_')
        self.cache_path = os.path.join(cache_base, f'{safe_coin}_cache.csv')

    def load_data(self, force_full=False):
        """
        智能数据加载:
        1. 检查本地缓存文件是否存在
        2. 若存在 → 加载缓存 + 仅下载最新增量数据 + 合并 + 更新缓存
        3. 若不存在或 force_full=True → 全量下载 + 保存缓存
        """
        t0 = time.time()
        cached_df = None
        
        # --- Step 1: 尝试加载缓存 ---
        if not force_full and os.path.exists(self.cache_path):
            try:
                cached_df = pd.read_csv(self.cache_path, parse_dates=['Date'])
                cached_df['Date'] = pd.to_datetime(cached_df['Date']).dt.tz_localize(None).dt.normalize()
                cached_df['Close'] = cached_df['Close'].astype(float)
                cached_df = cached_df.dropna(subset=['Date', 'Close'])
                last_date = cached_df['Date'].max()
                cache_rows = len(cached_df)
                print(f"📂 缓存命中: {self.cache_path}")
                print(f"   缓存数据: {cached_df['Date'].min().date()} → {last_date.date()} ({cache_rows:,} 行)")
            except Exception as e:
                print(f"⚠️ 缓存读取失败 ({e})，将全量下载")
                cached_df = None
        
        # --- Step 2: 下载数据 (增量或全量) ---
        if cached_df is not None:
            # 增量模式: 从缓存最后一天的前一天开始下载 (overlap 1天以处理盘中更新)
            fetch_start = (last_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
            print(f"⚡ 增量下载: {fetch_start} → 今天")
            new_df = yf.download(self.coin, start=fetch_start, progress=False)
            
            if not new_df.empty:
                new_df = new_df.reset_index()
                if isinstance(new_df.columns, pd.MultiIndex):
                    new_df.columns = new_df.columns.get_level_values(0)
                new_df = new_df[['Date', 'Close']].copy()
                new_df.columns = ['Date', 'Close']
                new_df['Date'] = pd.to_datetime(new_df['Date']).dt.tz_localize(None).dt.normalize()
                new_df['Close'] = new_df['Close'].astype(float)
                new_df = new_df.dropna()
                new_rows = len(new_df)
                
                # 合并: 用新数据覆盖重叠日期
                merged = pd.concat([cached_df, new_df], ignore_index=True)
                merged = merged.drop_duplicates(subset='Date', keep='last')
                merged = merged.sort_values('Date').reset_index(drop=True)
                self.prices = merged
                
                added = len(merged) - cache_rows
                print(f"   新增 {added} 行, 更新 {new_rows - added} 行")
            else:
                print("   无新数据, 使用缓存")
                self.prices = cached_df
        else:
            # 全量模式
            print(f"🌐 全量下载 {self.coin} (从 2010-07-18 至今)...")
            df = yf.download(self.coin, start='2010-07-18', progress=False)
            if df.empty:
                raise ValueError("Failed to fetch data.")
            df = df.reset_index()
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df[['Date', 'Close']].copy()
            df.columns = ['Date', 'Close']
            df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None).dt.normalize()
            df['Close'] = df['Close'].astype(float)
            self.prices = df.dropna().reset_index(drop=True)
            print(f"   下载完成: {len(self.prices):,} 行")
        
        # --- Step 3: 保存/更新缓存 ---
        try:
            self.prices.to_csv(self.cache_path, index=False)
            print(f"💾 缓存已更新: {self.cache_path}")
        except Exception as e:
            print(f"⚠️ 缓存写入失败 ({e})，不影响分析")
        
        elapsed = time.time() - t0
        print(f"⏱️ 数据加载完成: {elapsed:.1f}s ({len(self.prices):,} 行)")
        
        self._fit_model()
        return self.prices

    def _fit_model(self):
        """执行幂律拟合: log10(Price) = w × log10(Days) + b"""
        df = self.prices.copy()
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df = df[(df['Days'] > 0) & (df['Close'] > 1)].copy()
        
        self.dates = df['Date']
        x = np.log10(df['Days'].values).reshape(-1, 1)
        self.ydata = np.log10(df['Close'].values)
        
        model = LinearRegression().fit(x, self.ydata)
        self.w = model.coef_[0]
        self.b = model.intercept_
        self.predicted_ydata = model.predict(x)
        self.r2 = r2_score(self.ydata, self.predicted_ydata)
        
        print(f"Model Fitted: w={self.w:.4f}, b={self.b:.4f}, R²={self.r2:.4f}")

    def calculate_indicators(self, current_price=None):
        """计算当前的 ahr999 指数"""
        if self.prices is None:
            self.load_data()
            
        latest_data = self.prices.iloc[-1]
        p_now = current_price if current_price is not None else float(latest_data['Close'])
        date_now = latest_data['Date']
        
        # 过去 199 天价格和（不包含最新一天，与 p_now 合计正好为 200 天）
        past_199_sum = self.prices.iloc[-200:-1]['Close'].sum()
        ma200 = (past_199_sum + p_now) / 200
        
        days = (date_now - self.start_date).days
        if days <= 0:
            raise ValueError(f"Date {date_now} is before start_date {self.start_date}")
        fit_price = 10 ** (self.w * math.log10(days) + self.b)
        
        ahr999 = (p_now / ma200) * (p_now / fit_price)
        
        return {
            "date": date_now,
            "current_price": p_now,
            "ma200": ma200,
            "fit_price": fit_price,
            "ahr999": ahr999
        }

    def get_threshold(self, target_ahr999):
        """根据目标 ahr999 逆推币价"""
        latest_data = self.prices.iloc[-1]
        days = (latest_data['Date'] - self.start_date).days
        if days <= 0:
            return np.nan
        fit_price = 10 ** (self.w * math.log10(days) + self.b)
        # 前 199 天历史价格和（不包含当天）
        sum199 = self.prices.iloc[-200:-1]['Close'].sum()
        
        a = 200
        b = - (target_ahr999 * fit_price)
        c = - (target_ahr999 * fit_price * sum199)
        
        delta = b**2 - 4*a*c
        if delta < 0: return np.nan
        return (-b + math.sqrt(delta)) / (2 * a)

    def build_history(self):
        """构建完整的 AHR999 历史序列（用于回测和绘图）"""
        df = self.prices.copy()
        df['MA200'] = df['Close'].rolling(200).mean()
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df = df[df['Days'] > 0].copy()
        df['FitPrice'] = 10 ** (self.w * np.log10(df['Days']) + self.b)
        df['AHR999'] = (df['Close'] / df['MA200']) * (df['Close'] / df['FitPrice'])
        return df.dropna()

    def calculate_snr(self, window=30):
        """计算 AHR999 信噪比 (SNR in dB)"""
        df = self.build_history()
        ahr_tail = df['AHR999'].tail(window)
        trend = ahr_tail.rolling(5).mean().dropna()
        noise = (ahr_tail - ahr_tail.rolling(5).mean()).dropna()
        if noise.var() <= 0 or trend.var() <= 0:
            return 0.0
        return round(10 * math.log10(trend.var() / (noise.var() + 1e-9)), 1)

    def confidence_grade(self):
        """模型置信度评级 (与网站 haowu999_summary.py 一致)"""
        snr = self.calculate_snr()
        if self.r2 >= 0.7 and snr >= 6:
            return 'A+', snr
        elif self.r2 >= 0.5 and snr >= 3:
            return 'A', snr
        elif self.r2 >= 0.2 and snr >= 0:
            return 'B', snr
        else:
            return 'C', snr

    def clear_cache(self):
        """手动清除缓存文件，强制下次全量下载"""
        if os.path.exists(self.cache_path):
            os.remove(self.cache_path)
            print(f"🗑️ 缓存已清除: {self.cache_path}")
        else:
            print("无缓存文件")

print('✅ Haowu999 class loaded (with incremental cache).')


In [ ]:
class TestHaowu999(unittest.TestCase):
    def test_logic(self):
        import pandas as pd
        # Mock data with 250 days to ensure 200-day rolling window is full
        mock_data = pd.DataFrame({
            "Date": [pd.to_datetime("2020-01-01") + pd.Timedelta(days=i) for i in range(250)],
            "Close": [10000.0 + i * 10 for i in range(250)]
        })
        tester = Haowu999(start_date='2009-01-03')
        tester.prices = mock_data
        tester.w, tester.b, tester.r2 = 5.6, -16.0, 0.92
        
        # Test default indicators vs rolling mean
        indicators = tester.calculate_indicators()
        expected_ma200 = mock_data['Close'].iloc[-200:].mean()
        self.assertAlmostEqual(indicators['ma200'], expected_ma200, places=4)
        
        # Test threshold calculation and reverse indicator consistency
        p_45 = tester.get_threshold(0.45)
        res = tester.calculate_indicators(current_price=p_45)
        self.assertAlmostEqual(res['ahr999'], 0.45, places=5)

        # Test sell threshold
        p_sell = tester.get_threshold(5.0)
        self.assertIsNotNone(p_sell)
        self.assertTrue(p_sell > p_45)

        print("✅ All unit tests passed.")

suite = unittest.TestLoader().loadTestsFromTestCase(TestHaowu999)
unittest.TextTestRunner(verbosity=1).run(suite)


## 📊 1. 核心分析结果

In [ ]:
# 首次运行: 全量下载 ~15年数据并缓存
# 再次运行: 秒级增量更新 ⚡
# 如需强制全量重载: calc.load_data(force_full=True)
# 如需清除缓存:     calc.clear_cache()

calc = Haowu999()
calc.load_data()
stats = calc.calculate_indicators()

# === 买入/卖出信号线 ===
p_045 = calc.get_threshold(0.45)
p_120 = calc.get_threshold(1.2)
p_sell = calc.get_threshold(5.0)

# === SNR + Confidence Grade ===
grade, snr = calc.confidence_grade()

print("\n" + "═"*50)
print(f"  ₿ BTC 深度分析报告 — {stats['date'].date()}")
print("═"*50)
print(f"  当前价格:      ${stats['current_price']:>12,.2f}")
print(f"  200日均线:     ${stats['ma200']:>12,.2f}")
print(f"  拟合公允价:    ${stats['fit_price']:>12,.2f}")
print(f"  AHR999 指数:   {stats['ahr999']:>12.4f}")
print("─"*50)
print(f"  抄底线 (0.45): ${p_045:>12,.2f}")
print(f"  定投线 (1.20): ${p_120:>12,.2f}")
print(f"  卖出线 (5.00): ${p_sell:>12,.2f}")
print("─"*50)
print(f"  模型 R²:        {calc.r2:.4f}")
print(f"  信噪比 (SNR):   {snr:.1f} dB")
print(f"  模型置信度:     {grade}")
print("═"*50)

# Signal interpretation
ahr = stats['ahr999']
if ahr < 0.45:
    signal = "💎 DEEP VALUE — 极度低估，历史级抄底区间"
elif ahr < 0.85:
    signal = "🟢 ACCUMULATION — 价值区间，适合定投"
elif ahr < 1.2:
    signal = "✅ WATCHLIST — 合理估值，观察为主"
elif ahr < 2.5:
    signal = "☕ EXTENDED — 偏高估，谨慎追涨"
elif ahr < 5.0:
    signal = "⚠️ STRETCHED — 高位区间，考虑分批止盈"
else:
    signal = "🔥 EUPHORIA — 极度狂热，强烈建议减仓"

print(f"\n  📡 信号判定: {signal}")
print("═"*50)

## 🔬 2. 模型准确度审计

In [ ]:
# === 深度准确度审计 ===
import seaborn as sns

residuals = calc.ydata - calc.predicted_ydata
rmse = np.sqrt(mean_squared_error(calc.ydata, calc.predicted_ydata))
r2 = r2_score(calc.ydata, calc.predicted_ydata)

# MAPE (原始价格空间)
actual_prices = 10**calc.ydata
predicted_prices = 10**calc.predicted_ydata
mape = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Model Accuracy Audit', fontsize=14, fontweight='bold')

# 左图：残差随时间分布
axes[0].plot(calc.dates, residuals, color="purple", alpha=0.5, linewidth=0.8)
axes[0].axhline(y=0, color="black", linestyle="--", alpha=0.5)
axes[0].fill_between(calc.dates, residuals, 0, alpha=0.1, color='purple')
axes[0].set_title(f"Residuals Over Time (RMSE: {rmse:.4f})")
axes[0].set_ylabel("Log₁₀ Error")

# 右图：残差分布直方图
sns.histplot(residuals, kde=True, ax=axes[1], color="green", bins=50)
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1].set_title("Error Distribution (Bell Curve Test)")

plt.tight_layout()
plt.show()

print(f"\n--- 拟合质量报告 ---")
print(f"拟合优度 (R²):       {r2:.4f}  (越接近 1 越准)")
print(f"平均预测误差 (MAPE): {mape:.1f}%   (越小越准)")
print(f"对数均方根误差:      {rmse:.4f}")
print(f"残差均值:            {np.mean(residuals):.6f}  (应接近 0)")
print(f"残差标准差:          {np.std(residuals):.4f}")

if r2 > 0.9:
    print("\n结论: 🌟 模型极其稳健 — BTC 幂律增长规律性极强，信号极具参考价值。")
elif r2 > 0.8:
    print("\n结论: ✅ 模型较为可靠，具有良好参考价值。")
else:
    print("\n结论: ⚠️ 模型波动较大，建议仅作为辅助参考。")

## 🔄 3. 价格走势 + 减半周期叠加

In [ ]:
# === 增强版绘图 + 减半周期叠加 ===
df_hist = calc.build_history()

fig, axes = plt.subplots(2, 1, figsize=(16, 13))

# --- 上图: 价格 vs 回归线 ---
axes[0].plot(calc.dates, 10**calc.ydata, label='Actual Price', alpha=0.7, linewidth=1)
axes[0].plot(calc.dates, 10**calc.predicted_ydata, label='Power-Law Fit', 
             color='red', linewidth=2, linestyle='--')
axes[0].set_yscale('log')
axes[0].set_title(f"Bitcoin Price vs Power-Law Regression (R²={calc.r2:.4f})", fontsize=13)
axes[0].set_ylabel('Price (USD, log scale)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# 叠加减半事件
for date_str, label in HALVING_EVENTS:
    hd = pd.to_datetime(date_str)
    if hd >= calc.dates.iloc[0]:
        axes[0].axvline(x=hd, color='orange', linestyle=':', alpha=0.7, linewidth=1.5)
        axes[0].text(hd, axes[0].get_ylim()[0]*2, f' {label}', 
                     rotation=90, fontsize=7, color='darkorange', va='bottom')

# --- 下图: AHR999 历史 + 信号区间 ---
axes[1].plot(df_hist['Date'], df_hist['AHR999'], label='AHR999', 
             color='royalblue', linewidth=1, alpha=0.8)
axes[1].axhline(y=0.45, color='red', linestyle='--', linewidth=1.5, label='0.45 (抄底线)', alpha=0.8)
axes[1].axhline(y=1.2, color='green', linestyle='--', linewidth=1.5, label='1.20 (定投线)', alpha=0.8)
axes[1].axhline(y=5.0, color='darkred', linestyle='-.', linewidth=1.5, label='5.00 (卖出线)', alpha=0.6)

# 填充颜色区间 (使用大于 0 的正数避免 log 轴警告)
axes[1].axhspan(0.05, 0.45, alpha=0.08, color='red', label='Deep Value Zone (<0.45)')
axes[1].axhspan(0.45, 1.2, alpha=0.08, color='green', label='Accumulation Zone (0.45-1.20)')
axes[1].axhspan(5.0, 50, alpha=0.08, color='orange', label='Sell Zone (>5.00)')

axes[1].set_yscale('log')
axes[1].set_ylim(bottom=0.1, top=30)
axes[1].set_title("Historical AHR999 Trend with Signal Zones", fontsize=13)
axes[1].set_ylabel('AHR999 (log scale)')
axes[1].legend(loc='upper left', fontsize=8)
axes[1].grid(True, alpha=0.3)

# 叠加减半事件到 AHR999 图
for date_str, label in HALVING_EVENTS:
    hd = pd.to_datetime(date_str)
    if hd >= df_hist['Date'].iloc[0]:
        axes[1].axvline(x=hd, color='orange', linestyle=':', alpha=0.6, linewidth=1.5)
        axes[1].text(hd, 0.12, f' {label.split("(")[0].strip()}', 
                     rotation=90, fontsize=7, color='darkorange', va='bottom')

plt.tight_layout()
plt.show()

print("💡 观察：每次减半后 12-18 个月，AHR999 通常会飙升至 5.0 以上（狂热区间），随后回落。")


## 💰 4. 历史回测模拟

In [ ]:
# === 策略回测 ===
df_bt = calc.build_history().copy()

# --- 策略 A: AHR999 < 0.45 时每次买入 $100 ---
df_bt['buy_dip'] = (df_bt['AHR999'] < 0.45).astype(int) * 100

# --- 策略 B: AHR999 < 1.2 时每次买入 $100 ---
df_bt['buy_dca'] = (df_bt['AHR999'] < 1.2).astype(int) * 100

# --- 策略 C: 无脑每天定投 $100 ---
df_bt['buy_blind'] = 100

strategies = [
    ('💎 抄底策略 (AHR999 < 0.45)', 'buy_dip'),
    ('🟢 定投策略 (AHR999 < 1.20)', 'buy_dca'),
    ('📅 无脑定投 (每天 $100)', 'buy_blind'),
]

current_price = df_bt['Close'].iloc[-1]

print("═"*60)
print("  ₿ AHR999 策略历史回测报告")
print(f"  回测区间: {df_bt['Date'].iloc[0].date()} → {df_bt['Date'].iloc[-1].date()}")
print(f"  当前 BTC 价格: ${current_price:,.2f}")
print("═"*60)

backtest_results = []

for name, col in strategies:
    total_invested = df_bt[col].sum()
    shares_bought = (df_bt[col] / df_bt['Close']).sum()
    current_value = shares_bought * current_price
    roi = (current_value / total_invested - 1) * 100 if total_invested > 0 else 0
    buy_days = (df_bt[col] > 0).sum()
    total_days = len(df_bt)
    
    print(f"\n  {name}")
    print(f"    买入天数:   {buy_days:>6,} / {total_days:,} ({buy_days/total_days*100:.1f}%)")
    print(f"    总投入:     ${total_invested:>12,.0f}")
    print(f"    当前市值:   ${current_value:>12,.0f}")
    print(f"    总收益率:   {roi:>11,.1f}%")
    print(f"    收益倍数:   {current_value/total_invested:>11.1f}x" if total_invested > 0 else "")
    
    backtest_results.append({
        'strategy': name.split('(')[0].strip(),
        'invested': total_invested,
        'value': current_value,
        'roi': roi,
        'buy_pct': buy_days/total_days*100
    })

print("\n" + "═"*60)
print("  💡 结论: AHR999 抄底策略通常在用最少资金的情况下获得最高收益率，")
print("     因为它只在极度低估时出手，大幅降低了平均持仓成本。")
print("═"*60)

# --- 可视化对比 ---
fig, ax = plt.subplots(figsize=(10, 5))
labels = [r['strategy'] for r in backtest_results]
rois = [r['roi'] for r in backtest_results]
colors = ['#ff6b6b', '#51cf66', '#339af0']
bars = ax.bar(labels, rois, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
for bar, roi in zip(bars, rois):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            f'{roi:,.0f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('AHR999 Strategy Backtest — ROI Comparison', fontsize=13)
ax.set_ylabel('Total Return (%)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 🔁 5. 多周期幂律拟合对比

In [ ]:
# === 多周期幂律拟合对比 ===
# 验证 BTC 幂律是否在衰减
# 科学方法：时间原点 t0 始终固定为创世日 (2009-01-03)，对不同历史周期切片进行幂律拟合

windows = [
    ("Full History (Genesis)", "2009-01-03"),
    ("Post-2nd Halving (2016+)", "2016-07-09"),
    ("Post-3rd Halving (2020+)", "2020-05-11"),
    ("Post-4th Halving (2024+)", "2024-04-19"),
]

print("═"*68)
print("  ₿ 多周期幂律拟合对比 — 斜率 w 是否在衰减？")
print("═"*68)
print(f"  {'周期':<28} {'斜率 w':>8} {'截距 b':>8} {'R²':>8} {'样本天数':>8}")
print("─"*68)

multi_results = []
base_prices = calc.prices.copy()
genesis_date = pd.to_datetime('2009-01-03')
base_prices['Days'] = (base_prices['Date'] - genesis_date).dt.days

for label, start_str in windows:
    try:
        sub_df = base_prices[(base_prices['Date'] >= pd.to_datetime(start_str)) & (base_prices['Days'] > 0) & (base_prices['Close'] > 1)].copy()
        if len(sub_df) < 30:
            print(f"  {label:<28} ⚠️ 样本量过少 ({len(sub_df)} 天)")
            continue
            
        x_sub = np.log10(sub_df['Days'].values).reshape(-1, 1)
        y_sub = np.log10(sub_df['Close'].values)
        model_sub = LinearRegression().fit(x_sub, y_sub)
        w_sub = model_sub.coef_[0]
        b_sub = model_sub.intercept_
        r2_sub = r2_score(y_sub, model_sub.predict(x_sub))
        
        print(f"  {label:<28} {w_sub:>8.4f} {b_sub:>8.4f} {r2_sub:>8.4f} {len(sub_df):>8,}")
        multi_results.append({'label': label, 'w': w_sub, 'b': b_sub, 'r2': r2_sub, 'count': len(sub_df)})
    except Exception as e:
        print(f"  {label:<28} ⚠️ 拟合异常: {e}")

print("═"*68)

if len(multi_results) >= 2:
    w_full = multi_results[0]['w']
    w_recent = multi_results[1]['w'] # 采用 2016+ 代表中长期趋势
    print("\n  📉 幂律斜率趋势分析:")
    print(f"     全历史斜率: {w_full:.4f}  →  2016+ 斜率: {w_recent:.4f} (衰减: {(w_recent/w_full-1)*100:+.1f}%)")
    print("     BTC 超额增长斜率随网络体量扩大呈现温和递减，符合大类资产成熟化规律。")


## 🖥️ 6. Plotly 交互式仪表盘

In [ ]:
# === Plotly 交互式仪表盘 ===
df_hist = calc.build_history()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(
        f'Bitcoin Price vs Power-Law Regression (R²={calc.r2:.4f})',
        'AHR999 Index with Signal Zones'
    ),
    vertical_spacing=0.08,
    row_heights=[0.55, 0.45]
)

# --- Row 1: Price ---
fig.add_trace(go.Scatter(
    x=calc.dates, y=10**calc.ydata, name='BTC Price',
    line=dict(color='royalblue', width=1.5), opacity=0.8
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=calc.dates, y=10**calc.predicted_ydata, name='Power-Law Fit',
    line=dict(color='red', width=2, dash='dash')
), row=1, col=1)

# --- Row 2: AHR999 ---
fig.add_trace(go.Scatter(
    x=df_hist['Date'], y=df_hist['AHR999'], name='AHR999',
    line=dict(color='#667eea', width=1.5), opacity=0.9
), row=2, col=1)

# Signal lines
for val, color, name_label in [(0.45, 'red', '抄底 0.45'), (1.2, 'green', '定投 1.20'), (5.0, 'darkred', '卖出 5.00')]:
    fig.add_hline(y=val, line_dash='dash', line_color=color, line_width=1.5,
                  annotation_text=name_label, annotation_position='right',
                  row=2, col=1)

# Halving events
for date_str, label in HALVING_EVENTS:
    hd = pd.to_datetime(date_str)
    for row in [1, 2]:
        fig.add_vline(x=hd, line_dash='dot', line_color='orange', line_width=1,
                      opacity=0.5, row=row, col=1)

fig.update_yaxes(type='log', row=1, col=1, title_text='Price (USD)')
fig.update_yaxes(type='log', row=2, col=1, title_text='AHR999')
fig.update_xaxes(title_text='Date', row=2, col=1)

fig.update_layout(
    height=800,
    template='plotly_dark',
    title=dict(text='₿ Bitcoin AHR999 Interactive Dashboard — Haowu999', font=dict(size=16)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified'
)

fig.show()
print("💡 提示: 鼠标悬浮查看具体日期和价格，拖动可缩放特定区间。")

## ⛓️ 7. 链上数据叠加（MVRV 估值）

In [ ]:
# === 链上估值分析（离线确定性市值 + MVRV / Mayer Multiple 模型）===
# 基于比特币确定性减半供给模型计算历史真实流通量，免除外部 API 限流与付费门槛
def get_btc_circulating_supply(dates):
    """根据比特币减半规则计算精确的历史流通量 (Supply Curve)"""
    h0 = pd.to_datetime('2009-01-03')
    h1 = pd.to_datetime('2012-11-28') # 50 -> 25 BTC/block (7200 BTC/day)
    h2 = pd.to_datetime('2016-07-09') # 25 -> 12.5 BTC/block (3600 BTC/day)
    h3 = pd.to_datetime('2020-05-11') # 12.5 -> 6.25 BTC/block (1800 BTC/day)
    h4 = pd.to_datetime('2024-04-19') # 6.25 -> 3.125 BTC/block (900 BTC/day)
    
    supplies = []
    for d in dates:
        d = pd.to_datetime(d)
        supply = 0.0
        if d > h0:
            supply += min((d - h0).days, (h1 - h0).days) * 7200
        if d > h1:
            supply += min((d - h1).days, (h2 - h1).days) * 3600
        if d > h2:
            supply += min((d - h2).days, (h3 - h2).days) * 1800
        if d > h3:
            supply += min((d - h3).days, (h4 - h3).days) * 900
        if d > h4:
            supply += (d - h4).days * 450
        supplies.append(min(supply, 21000000.0))
    return np.array(supplies)

print("Calculating on-chain valuation & Market Cap...")
mc_df = calc.prices.copy()
mc_df['Supply'] = get_btc_circulating_supply(mc_df['Date'])
mc_df['MarketCap'] = mc_df['Close'] * mc_df['Supply']
mc_df['MA365_Cap'] = mc_df['MarketCap'].rolling(365).mean()
mc_df['MVRV_approx'] = mc_df['MarketCap'] / mc_df['MA365_Cap']
mc_df = mc_df.dropna().reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=(16, 6))

color1 = 'royalblue'
ax1.plot(mc_df['Date'], mc_df['MVRV_approx'], color=color1, linewidth=1.2, alpha=0.85, label='MVRV 近似 (Market Cap / MA365)')
ax1.axhline(y=1.0, color='green', linestyle='--', alpha=0.6, label='MVRV = 1.0 (公允估值基准)')
ax1.axhline(y=2.5, color='orange', linestyle='--', alpha=0.6, label='MVRV = 2.5 (高估预警)')
ax1.axhline(y=3.5, color='red', linestyle='--', alpha=0.6, label='MVRV = 3.5 (历史狂热顶部)')
ax1.axhline(y=0.7, color='darkgreen', linestyle='--', alpha=0.6, label='MVRV = 0.7 (历史深度抄底)')
ax1.set_ylabel('MVRV Ratio (近似)', color=color1)
ax1.set_title('Bitcoin On-Chain Valuation: MVRV Proxy (Market Cap vs 365D Realized Proxy)', fontsize=13)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.25)

# 叠加减半线
for date_str, label in HALVING_EVENTS:
    hd = pd.to_datetime(date_str)
    if hd >= mc_df['Date'].iloc[0]:
        ax1.axvline(x=hd, color='orange', linestyle=':', alpha=0.6, linewidth=1.2)

plt.tight_layout()
plt.show()

current_mvrv = mc_df['MVRV_approx'].iloc[-1]
current_supply = mc_df['Supply'].iloc[-1]
current_mcap = mc_df['MarketCap'].iloc[-1]

print(f"\n  当前 BTC 流通量:  {current_supply:,.0f} BTC")
print(f"  当前全网总市值:  ${current_mcap:,.0f}")
print(f"  当前 MVRV (近似): {current_mvrv:.2f}")

if current_mvrv < 0.7:
    print("  📊 链上判定: 💎 深度低估 — 市场极度恐慌，极高赔率区间")
elif current_mvrv < 1.0:
    print("  📊 链上判定: 🟢 低估区间 — 大多数持有者处于平均持仓成本附近")
elif current_mvrv < 2.0:
    print("  📊 链上判定: ✅ 合理估值 — 市场情绪健康中性")
elif current_mvrv < 3.0:
    print("  📊 链上判定: ☕ 偏高估 — 获利盘增加，适度关注回撤风险")
else:
    print("  📊 链上判定: 🔥 过热 — 高位获利了结风险巨大")


## 📋 8. 综合总结

In [ ]:
# === 综合总结面板 ===
stats = calc.calculate_indicators()
p_045 = calc.get_threshold(0.45)
p_120 = calc.get_threshold(1.2)
p_sell = calc.get_threshold(5.0)
grade, snr = calc.confidence_grade()

ahr = stats['ahr999']
if ahr < 0.45:
    signal = "💎 DEEP VALUE — 极度低估，历史级抄底区间"
elif ahr < 0.85:
    signal = "🟢 ACCUMULATION — 价值区间，适合定投"
elif ahr < 1.2:
    signal = "✅ WATCHLIST — 合理估值，观察为主"
elif ahr < 2.5:
    signal = "☕ EXTENDED — 偏高估，谨慎追涨"
elif ahr < 5.0:
    signal = "⚠️ STRETCHED — 高位区间，考虑分批止盈"
else:
    signal = "🔥 EUPHORIA — 极度狂热，强烈建议减仓"

print("\n" + "═"*60)
print("              ₿ Bitcoin AHR999 综合研究报告")
print("═"*60)
print(f"  日期:            {str(stats['date'].date()):>38}")
print(f"  当前价格:        ${stats['current_price']:>37,.2f}")
print(f"  200日均线:       ${stats['ma200']:>37,.2f}")
print(f"  拟合公允价:      ${stats['fit_price']:>37,.2f}")
print(f"  AHR999 指数:     {stats['ahr999']:>38.4f}")
print("─"*60)
print(f"  抄底线 (0.45):   ${p_045:>37,.2f}")
print(f"  定投线 (1.20):   ${p_120:>37,.2f}")
print(f"  卖出线 (5.00):   ${p_sell:>37,.2f}")
print("─"*60)
print(f"  模型 R²:         {calc.r2:>38.4f}")
print(f"  信噪比 (SNR):    {snr:>35.1f} dB")
print(f"  模型置信度:      {grade:>38}")
print("═"*60)
print(f"  📡 信号判定: {signal}")
print("═"*60)

print("\n📖 更多资产分析请访问: https://wuhao007.github.io/haowu999/")
